# Tutorial 8 - Python API Workflow

This tutorial assembles the public Python pieces into a small automation workflow without using the CLI.


In [ ]:
from __future__ import annotations

import csv
import json
import os
import shutil
import sqlite3
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any

RUN_LIVE = os.getenv("BRAIN_SIM_RUN_LIVE") == "1"
CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / "examples").exists() else CWD.parent if CWD.name == "examples" else CWD
EXAMPLE_DIR = ROOT / "examples"
DATA_DIR = EXAMPLE_DIR / "data"
EXPECTED_DIR = EXAMPLE_DIR / "expected"
RUNS_DIR = EXAMPLE_DIR / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Repository root: {ROOT}")
print(f"Live execution enabled: {RUN_LIVE}")


In [ ]:
@dataclass(frozen=True)
class TutorialRateLimitState:
    limit: int | None
    remaining: int | None
    reset_seconds: int | None


@dataclass(frozen=True)
class TutorialSubmitResult:
    status_code: int
    location: str
    body: Any
    headers: dict[str, str]
    rate_limit: TutorialRateLimitState


@dataclass(frozen=True)
class TutorialPollResult:
    status: str
    body: Any
    events: list[dict[str, Any]]


class CompleteFakeBrainClient:
    def __init__(self, alpha_prefix: str = "tutorial-alpha", location_prefix: str = "/simulations/tutorial") -> None:
        self.alpha_prefix = alpha_prefix
        self.location_prefix = location_prefix
        self.submit_count = 0
        self.locations: dict[str, list[str]] = {}

    def submit(self, payload: dict[str, Any] | list[dict[str, Any]]) -> TutorialSubmitResult:
        self.submit_count += 1
        payloads = payload if isinstance(payload, list) else [payload]
        alpha_ids = [f"{self.alpha_prefix}-{self.submit_count + offset}" for offset in range(len(payloads))]
        location = f"{self.location_prefix}-{self.submit_count}"
        self.locations[location] = alpha_ids
        return TutorialSubmitResult(
            status_code=201,
            location=location,
            body={"id": location.rsplit("/", 1)[-1]},
            headers={"Location": location, "X-Ratelimit-Remaining": "999"},
            rate_limit=TutorialRateLimitState(limit=None, remaining=999, reset_seconds=None),
        )

    def poll(self, location: str, *, timeout_seconds: float) -> TutorialPollResult:
        alpha_ids = self.locations[location]
        body: Any = {"alpha": alpha_ids[0]} if len(alpha_ids) == 1 else {"alphas": alpha_ids}
        return TutorialPollResult(status="complete", body=body, events=[{"body": body}])

    def fetch_alpha(self, alpha_id: str) -> dict[str, Any]:
        index = int(alpha_id.rsplit("-", 1)[-1])
        return {
            "id": alpha_id,
            "status": "UNSUBMITTED",
            "is": {
                "sharpe": f"{1.0 + index / 10:.2f}",
                "fitness": f"{0.5 + index / 10:.2f}",
                "returns": f"{0.03 + index / 100:.2f}",
                "turnover": f"{0.10 + index / 100:.2f}",
                "drawdown": f"{0.02 + index / 100:.2f}",
                "checks": [
                    {"name": "LOW_SHARPE", "result": "WARNING"} if index == 2 else {"name": "SELF_CORRELATION", "result": "PASS"}
                ],
            },
        }

    def fetch_recordset(self, alpha_id: str, recordset_name: str) -> dict[str, Any]:
        return {
            "alpha_id": alpha_id,
            "recordset": recordset_name,
            "rows": [
                {"date": "2026-01-01", "value": 0.01},
                {"date": "2026-01-02", "value": 0.02},
            ],
        }


## 1. Import The API Pieces

Use `BrainAuth` for cookie management, `read_excel_expressions` for input, `build_payload_record` for payloads, `BrainClient` for live HTTP access, `BatchRunner` for orchestration, and `RunStore` for artifacts.


In [ ]:
from brain_sim.auth import BrainAuth
from brain_sim.batch import BatchRunner
from brain_sim.client import BrainClient
from brain_sim.excel import read_excel_expressions
from brain_sim.payloads import build_payload_record
from brain_sim.results import RunStore

cookie_path = RUNS_DIR / "tutorial_08_api" / "cookies.json"
auth = BrainAuth(cookie_path=cookie_path)
live_client = BrainClient()
store = RunStore(RUNS_DIR / "tutorial_08_api" / "empty_store")
print(type(auth).__name__, type(live_client).__name__, store.run_dir.exists())


## 2. Build A Custom Offline Automation Function

The same function can accept `BrainClient()` after authentication. In this notebook it receives a fake client.


In [ ]:
def run_excel_queue(excel_path: Path, *, client: Any, run_dir: Path) -> dict[str, Any]:
    alpha_rows = read_excel_expressions(excel_path, sheet_name="alphas")
    records = []
    for index, alpha in enumerate(alpha_rows, start=1):
        record = build_payload_record(alpha)
        records.append(record.__class__(record.row_id, f"api-hash-{index}", record.payload, record.metadata))
    runner = BatchRunner(client, run_dir)
    return runner.run(records, batch_size="auto", poll_timeout_seconds=5, recordsets=["pnl"])


api_run_dir = RUNS_DIR / "tutorial_08_api" / "run"
shutil.rmtree(api_run_dir, ignore_errors=True)
result = run_excel_queue(DATA_DIR / "tutorial_08_api_alphas.xlsx", client=CompleteFakeBrainClient(alpha_prefix="api-alpha"), run_dir=api_run_dir)
print(result)
sorted(path.relative_to(api_run_dir) for path in api_run_dir.rglob("*") if path.is_file())


## 3. Optional Live Swap

After `brain-sim login`, you can load saved cookies into a `requests.Session`, construct `BrainClient(session=session)`, and pass that client into the same function. Keep the `BRAIN_SIM_RUN_LIVE=1` guard for any notebook-based live execution.
